# Lesson 3: Subqueries, CTEs, and Window Functions

**Week 2 · Data Engineering Course**

---

**Prerequisites:** Lab 2 complete, Lessons 1 and 2 read.

**What you will learn:**
- Subqueries — queries inside queries
- CTEs (Common Table Expressions) — named, reusable query blocks
- Window functions — aggregate without collapsing rows

**All SQL runs in pgAdmin → Tools → Query Tool → select `de_course`**

---

## 3.1 Subqueries

A subquery is a SELECT statement nested inside another query. The inner query runs first; its result is used by the outer query.

### Subquery in WHERE

The most common form — the subquery produces a value or list that the outer WHERE uses.

**Example — products that have been ordered at least once:**

```sql
SELECT product_name, category, price
FROM products
WHERE product_id IN (
    SELECT DISTINCT product_id
    FROM order_items
)
ORDER BY category, product_name;
```

**Example — customers who spent more than the average customer spend:**

```sql
SELECT
    c.first_name || ' ' || c.last_name AS customer,
    SUM(oi.quantity * oi.unit_price) AS total_spent
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
INNER JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.first_name, c.last_name
HAVING SUM(oi.quantity * oi.unit_price) > (
    SELECT AVG(customer_total)
    FROM (
        SELECT SUM(oi2.quantity * oi2.unit_price) AS customer_total
        FROM orders o2
        INNER JOIN order_items oi2 ON o2.order_id = oi2.order_id
        GROUP BY o2.customer_id
    ) avg_calc
)
ORDER BY total_spent DESC;
```

> This is a nested subquery — a subquery inside a subquery. It works, but there is a cleaner way: CTEs.

---

### Subquery in FROM (derived table)

A subquery in the FROM clause acts as a temporary table. You must give it an alias.

**Example — top 5 orders by total value:**

```sql
SELECT order_id, order_total
FROM (
    SELECT
        o.order_id,
        SUM(oi.quantity * oi.unit_price) AS order_total
    FROM orders o
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id
) order_totals
ORDER BY order_total DESC
LIMIT 5;
```

The inner query calculates a total per order. The outer query sorts and limits — something you cannot do directly with GROUP BY and LIMIT together.

---

## 3.2 CTEs — Common Table Expressions

A CTE is a named subquery defined at the top of a query using the WITH keyword. It makes complex queries readable by breaking them into named steps.

**Syntax:**

```sql
WITH cte_name AS (
    SELECT ...
)
SELECT *
FROM cte_name;
```

**Example — top 5 orders by value (rewritten as a CTE):**

```sql
WITH order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_date,
        SUM(oi.quantity * oi.unit_price) AS order_total
    FROM orders o
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id, o.customer_id, o.order_date
)
SELECT
    ot.order_id,
    c.first_name || ' ' || c.last_name AS customer,
    ot.order_date,
    ot.order_total
FROM order_totals ot
INNER JOIN customers c ON ot.customer_id = c.customer_id
ORDER BY ot.order_total DESC
LIMIT 5;
```

The CTE and the derived table produce the same result — but the CTE version is easier to read and easier to debug.

---

### Multiple CTEs

You can chain CTEs separated by commas. Each CTE can reference those defined before it.

**Example — customer spend vs average, in two clean steps:**

```sql
WITH customer_totals AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price) AS total_spent
    FROM orders o
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.customer_id
),
avg_spend AS (
    SELECT AVG(total_spent) AS average_spend
    FROM customer_totals
)
SELECT
    c.first_name || ' ' || c.last_name AS customer,
    ct.total_spent,
    ROUND(avs.average_spend, 2) AS avg_spend,
    CASE
        WHEN ct.total_spent > avs.average_spend THEN 'Above average'
        ELSE 'Below average'
    END AS spend_tier
FROM customer_totals ct
INNER JOIN customers c ON ct.customer_id = c.customer_id
CROSS JOIN avg_spend avs
ORDER BY ct.total_spent DESC;
```

> `CROSS JOIN avg_spend` attaches the single average row to every customer row — a clean way to compare each row against a scalar value.

---

## 3.3 Window Functions

Window functions compute a value **across a set of related rows** without collapsing them. Unlike GROUP BY, you keep all the original rows — the window function result appears as an extra column.

**The key difference:**

```sql
-- GROUP BY: collapses to one row per category
SELECT category, SUM(price) FROM products GROUP BY category;

-- Window function: keeps all rows, adds a running sum column
SELECT product_name, category, price,
       SUM(price) OVER (PARTITION BY category) AS category_total
FROM products;
```

**Anatomy of a window function:**

```sql
function_name() OVER (
    PARTITION BY column   -- optional: divides rows into groups
    ORDER BY column       -- optional: defines the order within the window
)
```

---

### ROW_NUMBER, RANK, DENSE_RANK

These assign a position number to each row within its partition.

| Function | Behaviour on ties |
|----------|-------------------|
| `ROW_NUMBER()` | Always unique — tied rows get different numbers |
| `RANK()` | Tied rows share a rank; next rank skips (1, 1, 3) |
| `DENSE_RANK()` | Tied rows share a rank; next rank does not skip (1, 1, 2) |

**Example — rank products within each category by price:**

```sql
SELECT
    product_name,
    category,
    price,
    RANK() OVER (PARTITION BY category ORDER BY price DESC) AS rank_in_category
FROM products
ORDER BY category, rank_in_category;
```

**Example — top 1 product per category by price:**

```sql
WITH ranked AS (
    SELECT
        product_name,
        category,
        price,
        RANK() OVER (PARTITION BY category ORDER BY price DESC) AS rnk
    FROM products
)
SELECT product_name, category, price
FROM ranked
WHERE rnk = 1;

---

### SUM OVER and running totals

`SUM() OVER (ORDER BY ...)` computes a **running total** — each row shows the cumulative sum up to that point.

**Example — running total of order revenue by date:**

```sql
WITH daily_revenue AS (
    SELECT
        o.order_date,
        SUM(oi.quantity * oi.unit_price) AS day_revenue
    FROM orders o
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY o.order_date
)
SELECT
    order_date,
    day_revenue,
    SUM(day_revenue) OVER (ORDER BY order_date) AS running_total
FROM daily_revenue
ORDER BY order_date;
```

**Example — rank customers by total spend:**

```sql
WITH customer_spend AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price) AS total_spent
    FROM orders o
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.customer_id
)
SELECT
    c.first_name || ' ' || c.last_name AS customer,
    cs.total_spent,
    RANK() OVER (ORDER BY cs.total_spent DESC) AS spend_rank
FROM customer_spend cs
INNER JOIN customers c ON cs.customer_id = c.customer_id
ORDER BY spend_rank;

---

### LAG and LEAD

`LAG()` accesses the value from the **previous** row in the window. `LEAD()` accesses the **next** row. Both are used for period-over-period comparisons.

**Example — month-over-month revenue change:**

```sql
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', o.order_date) AS month,
        SUM(oi.quantity * oi.unit_price)  AS revenue
    FROM orders o
    INNER JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'completed'
    GROUP BY DATE_TRUNC('month', o.order_date)
)
SELECT
    month,
    revenue,
    LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue,
    revenue - LAG(revenue) OVER (ORDER BY month) AS month_change
FROM monthly
ORDER BY month;
```

> `DATE_TRUNC('month', date)` truncates a date to the first day of its month — `2023-07-15` becomes `2023-07-01`. It is the standard way to group by month in PostgreSQL.

---

## 3.4 When to Use Each

| Tool | Use when |
|------|----------|
| **Subquery in WHERE** | Filtering based on values from another query |
| **Subquery in FROM** | You need a temporary table to query against |
| **CTE** | The subquery is used more than once, or the query has multiple logical steps |
| **Window function** | You need to rank, compare, or compute running values without losing individual rows |

> **Rule of thumb:** if a nested subquery is hard to read, rewrite it as a CTE. CTEs are not faster — they are just easier to understand and maintain.

---

## 3.5 Key Takeaways

1. A **subquery** is a SELECT inside another query. The inner query always runs first.
2. Subqueries in **WHERE** filter using a value or list from another table.
3. Subqueries in **FROM** create a temporary table — they must have an alias.
4. **CTEs** (`WITH name AS (...)`) are named subqueries. They make complex queries readable and debuggable.
5. Multiple CTEs are separated by commas. Each can reference earlier ones.
6. **Window functions** add a computed column without collapsing rows — unlike GROUP BY.
7. The `OVER (PARTITION BY ... ORDER BY ...)` clause defines the window: what rows each calculation sees.
8. `RANK()` and `ROW_NUMBER()` assign positions within a partition.
9. `SUM() OVER (ORDER BY ...)` produces a **running total**.
10. `LAG()` and `LEAD()` access the previous or next row — used for period-over-period comparisons.